# Tugas Pertemuan 9 - Data Science

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Materi:** Algoritma Klasifikasi (Bagian 1)  
**Dataset:** Breast Cancer Wisconsin dari `sklearn.datasets.load_breast_cancer()`

Notebook ini dibuat untuk memenuhi aktivitas hands-on Pertemuan 9. Fokus pengerjaan adalah membandingkan model Logistic Regression dan Decision Tree untuk kasus diagnosis kanker.

## 1. Import Library

Library yang digunakan mengikuti materi modul, yaitu `pandas`, `matplotlib`, dan `scikit-learn`. Di Google Colab library ini umumnya sudah tersedia.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

pd.set_option("display.max_columns", 40)

## 2. Load Dataset dan EDA Singkat

Dataset yang digunakan adalah Breast Cancer Wisconsin. Target dataset ini terdiri dari dua kelas: `0 = malignant` atau ganas, dan `1 = benign` atau jinak.

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Shape fitur:", X.shape)
print("Shape target:", y.shape)
print("\nNama kelas:", list(data.target_names))
print("\nDistribusi target (jumlah):")
print(y.value_counts().rename(index={0: "malignant", 1: "benign"}))
print("\nDistribusi target (proporsi):")
print(y.value_counts(normalize=True).round(3).rename(index={0: "malignant", 1: "benign"}))

display(X.head())

In [ ]:
# Korelasi fitur terhadap target
eda_df = X.copy()
eda_df["target"] = y

corr_target = (
    eda_df.corr(numeric_only=True)["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

print("10 fitur dengan korelasi terkuat terhadap target:")
display(corr_target.head(10).to_frame("korelasi_target"))

plt.figure(figsize=(9, 5))
corr_target.head(10).sort_values().plot(kind="barh", color="#2E74B5")
plt.title("10 Fitur dengan Korelasi Terkuat terhadap Target")
plt.xlabel("Nilai Korelasi")
plt.tight_layout()
plt.show()

## 3. Preprocessing

Data dibagi menjadi 80 persen data latih dan 20 persen data uji. Parameter `stratify=y` digunakan agar distribusi kelas pada data latih dan data uji tetap seimbang. StandardScaler diterapkan untuk Logistic Regression karena model ini sensitif terhadap skala fitur.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji:", X_test.shape[0])
print("\nDistribusi target pada data uji:")
print(y_test.value_counts().rename(index={0: "malignant", 1: "benign"}))

## 4. Model Logistic Regression

Logistic Regression digunakan untuk memprediksi kelas berdasarkan probabilitas. Walaupun namanya regression, model ini digunakan untuk klasifikasi.

In [ ]:
log_model = LogisticRegression(max_iter=5000, random_state=42)
log_model.fit(X_train_s, y_train)

y_pred_log = log_model.predict(X_test_s)
y_proba_log = log_model.predict_proba(X_test_s)[:, 1]

coef_df = (
    pd.DataFrame({
        "Fitur": X.columns,
        "Koefisien": log_model.coef_[0],
        "Abs_Koefisien": abs(log_model.coef_[0])
    })
    .sort_values("Abs_Koefisien", ascending=False)
)

print("Intercept:", round(log_model.intercept_[0], 3))
print("10 koefisien paling berpengaruh:")
display(coef_df.head(10)[["Fitur", "Koefisien"]])

## 5. Model Decision Tree

Decision Tree dibuat dengan `max_depth=4` agar pohon tidak terlalu dalam. Model ini tidak membutuhkan scaling karena pemisahan datanya berdasarkan aturan threshold pada fitur.

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

importance_df = (
    pd.DataFrame({
        "Fitur": X.columns,
        "Importance": tree_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print("10 fitur paling penting pada Decision Tree:")
display(importance_df.head(10))

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=["Malignant", "Benign"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Visualisasi Decision Tree")
plt.tight_layout()
plt.show()

## 6. Evaluasi dan Perbandingan Model

Metrik yang digunakan adalah Confusion Matrix, Accuracy, Precision, Recall, dan F1-Score. Untuk kasus diagnosis kanker, recall menjadi perhatian utama karena false negative berarti kasus kanker yang sebenarnya positif dapat terlewat.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n=== {name} ===")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Malignant", "Benign"]))
    
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1],
    }

results = [
    evaluate_model("Logistic Regression", y_test, y_pred_log),
    evaluate_model("Decision Tree", y_test, y_pred_tree),
]

result_df = pd.DataFrame(results).set_index("Model")
display(result_df.round(3))

In [ ]:
best_recall_model = result_df["Recall"].idxmax()
best_f1_model = result_df["F1-Score"].idxmax()

print("Model dengan Recall tertinggi:", best_recall_model)
print("Model dengan F1-Score tertinggi:", best_f1_model)

if result_df.loc["Logistic Regression", "Recall"] == result_df.loc["Decision Tree", "Recall"]:
    print("\nRecall kedua model sama. Untuk kasus seperti ini, pertimbangkan F1-Score, stabilitas model, dan kemudahan interpretasi.")
else:
    print(f"\nUntuk kasus diagnosis, {best_recall_model} lebih diprioritaskan karena memiliki Recall paling tinggi.")

print("\nAlasan:")
print("Recall penting karena model harus sebisa mungkin mendeteksi pasien yang benar-benar bermasalah.")
print("False Negative pada diagnosis kanker lebih berbahaya daripada False Positive, karena pasien sakit bisa dianggap sehat.")

## 7. Kesimpulan

Berdasarkan praktikum, Logistic Regression dan Decision Tree sama-sama dapat digunakan untuk klasifikasi diagnosis kanker. Logistic Regression bekerja dengan mengubah kombinasi fitur menjadi probabilitas melalui fungsi sigmoid, sedangkan Decision Tree membuat aturan keputusan berbentuk pohon.

Untuk kasus diagnosis kanker, metrik yang paling penting adalah recall. Alasannya, kesalahan false negative sangat berisiko karena pasien yang sebenarnya memiliki kanker dapat tidak terdeteksi. Accuracy saja tidak cukup karena bisa terlihat tinggi walaupun model masih melewatkan kasus positif yang penting.

Model terbaik dapat dilihat dari tabel evaluasi setelah notebook dijalankan. Jika recall kedua model sama, maka F1-Score dan stabilitas model dapat dijadikan pertimbangan tambahan.

## 8. Catatan Upload GitHub

Setelah semua sel dijalankan dari awal sampai akhir dan output sudah muncul, upload file notebook ini ke repositori GitHub. Pastikan visualisasi Decision Tree dan tabel evaluasi terlihat sebelum link dikirim ke Forum Diskusi 9 di LMS.